# USL Championship — Player Data

Outfield player and goalkeeper metrics using the ASA public API.

In [1]:
from typing import Final

import pandas as pd
from itscalledsoccer.client import AmericanSoccerAnalysis

pd.options.display.max_columns = 999
pd.set_option("expand_frame_repr", False)

asa_client = AmericanSoccerAnalysis()

LEAGUE: Final[str] = "uslc"
GK_MIN_MINUTES: Final[int] = 100  # minimum minutes to qualify a GK for rate stat analysis

## Data Fetch

In [2]:
players_merge = asa_client.get_players(leagues=[LEAGUE])[
    ["player_id", "player_name", "birth_date", "nationality", "height_ft", "height_in"]
].copy()

players_merge["height_ft"] = players_merge["height_ft"].astype("Int64")
players_merge["height_in"] = players_merge["height_in"].astype("Int64")

# Build HEIGHT string only where both components are present;
# Int64 NA values stringify to "<NA>" without this guard.
_has_ht = players_merge["height_ft"].notna() & players_merge["height_in"].notna()
players_merge["HEIGHT"] = (
    players_merge["height_ft"].astype(str)
    + "' "
    + players_merge["height_in"].astype(str)
    + '"'
).where(_has_ht)
players_merge = players_merge.drop(columns=["height_ft", "height_in"])

In [3]:
players_xg = asa_client.get_player_xgoals(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)
players_xp = asa_client.get_player_xpass(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)
players_ga_raw = asa_client.get_player_goals_added(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)
gk_xg = asa_client.get_goalkeeper_xgoals(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)
gk_ga_raw = asa_client.get_goalkeeper_goals_added(
    leagues=[LEAGUE], split_by_seasons=True, stage_name="Regular Season"
)

## Data Cleaning

Build team lookup, resolve `team_id` to abbreviation, and prepare xPass for joining.

In [4]:
teams = asa_client.get_teams(leagues=[LEAGUE])
team_map: dict[str, str] = dict(zip(teams["team_id"], teams["team_abbreviation"]))


def _resolve_team(val: str | list[str], team_map: dict[str, str]) -> str | float:
    """Map a team_id (or list of team_ids) to its abbreviation(s).

    Players who transferred mid-season arrive with a list of team_ids;
    those are joined with '/' (e.g. 'LOU/IND'). Unknown IDs fall back to
    the raw ID string. NaN/None values pass through unchanged.
    """
    if isinstance(val, list):
        return "/".join(team_map.get(v, v) for v in val)
    return team_map.get(val, val)


def _flatten_goals_added(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten an ASA goals-added frame's nested `data` column into wide form.

    The API returns one row per (player, season) with `data` holding a list
    of dicts — one per action_type. This explodes to long form, normalizes
    the dicts, then pivots to one column per (metric, action_type). Also
    adds total ga_raw / ga_aboveavg sums across all action types.
    """
    long = df.explode("data", ignore_index=True)
    long = long.join(pd.json_normalize(long.pop("data")))
    long["action_type"] = long["action_type"].str.lower()
    wide = long.pivot(
        index=["player_id", "season_name"],
        columns="action_type",
        values=["goals_added_raw", "goals_added_above_avg", "count_actions"],
    )
    metric_map = {
        "goals_added_raw": "ga_raw",
        "goals_added_above_avg": "ga_aboveavg",
        "count_actions": "ga_actions",
    }
    wide.columns = [f"{metric_map[m]}_{a}" for m, a in wide.columns]
    wide = wide.reset_index()
    raw_cols = [c for c in wide.columns if c.startswith("ga_raw_")]
    aboveavg_cols = [c for c in wide.columns if c.startswith("ga_aboveavg_")]
    wide["ga_raw_total"] = wide[raw_cols].sum(axis=1)
    wide["ga_aboveavg_total"] = wide[aboveavg_cols].sum(axis=1)
    ga_value_cols = [
        c for c in wide.columns
        if c.startswith(("ga_raw_", "ga_aboveavg_"))
    ]
    wide[ga_value_cols] = wide[ga_value_cols].round(3)
    return wide


players_xg["team_name"] = players_xg["team_id"].apply(_resolve_team, team_map=team_map)
players_xg = players_xg.drop(columns=["team_id"])

gk_xg["team_name"] = gk_xg["team_id"].apply(_resolve_team, team_map=team_map)
gk_xg = gk_xg.drop(columns=["team_id"])

# Drop columns from xPass that would create merge conflicts.
# - team_id: already resolved to team_name in xGoals
# - general_position: present in xGoals; xPass version is redundant
# - minutes_played: present in xGoals; xPass version is redundant
# count_games is unique to xPass (xGoals does not return it) and is kept.
players_xp = players_xp.drop(
    columns=["team_id", "general_position", "minutes_played"],
    errors="ignore",
)
players_xp["season_name"] = players_xp["season_name"].astype("Int64")

# Flatten Goals Added frames into wide per-action columns.
players_ga = _flatten_goals_added(players_ga_raw)
players_ga["season_name"] = players_ga["season_name"].astype("Int64")
gk_ga = _flatten_goals_added(gk_ga_raw)
gk_ga["season_name"] = gk_ga["season_name"].astype("Int64")

## Outfield Players

In [5]:
players = players_merge.merge(players_xg, on="player_id", how="left")
players = players[players["season_name"].notna()]

players["season_name"] = players["season_name"].astype("Int64")
players["minutes_played"] = players["minutes_played"].astype("Int64")

players["MinPerShot"] = (players["minutes_played"] / players["shots"]).round(1)
players["MinPerSOT"] = (players["minutes_played"] / players["shots_on_target"]).round(1)
players["MinPerG"] = (players["minutes_played"] / players["goals"]).round(1)
players["MinPerXG"] = (players["minutes_played"] / players["xgoals"]).round(2)
players["ShtPerSOT"] = (players["shots"] / players["shots_on_target"]).round(2)
players["ShtPerG"] = (players["shots"] / players["goals"]).round(1)
players["ShtPerXG"] = (players["shots"] / players["xgoals"]).round(2)
players["GPerShot"] = (players["goals"] / players["shots"]).round(2)
players["xGPerShot"] = (players["xgoals"] / players["shots"]).round(3)
players["MinPerAssist"] = (players["minutes_played"] / players["primary_assists"]).round(2)
players["MinPerXA"] = (players["minutes_played"] / players["xassists"]).round(2)

# Replace inf values that arise when a denominator is zero (e.g. 0 shots, 0 goals).
players = players.replace([float("inf"), float("-inf")], pd.NA)

players["xgoals"] = players["xgoals"].round(2)

In [6]:
players = players.merge(
    players_xp,
    on=["player_id", "season_name"],
    how="left",
)
players["PassAttemptsPerGame"] = (players["attempted_passes"] / players["count_games"]).round(2)

In [7]:
# Merge in flattened Goals Added (per-action + totals).
_ga_join = ["player_id", "season_name"]
_ga_extra = [c for c in players_ga.columns if c not in players.columns]
players = players.merge(
    players_ga[_ga_join + _ga_extra], on=_ga_join, how="left"
)

In [8]:
# Per-90 normalization — the standard rate basis in football analytics.
_per90 = 90 / players["minutes_played"]
players["G_p90"] = (players["goals"] * _per90).round(2)
players["xG_p90"] = (players["xgoals"] * _per90).round(2)
players["A_p90"] = (players["primary_assists"] * _per90).round(2)
players["xA_p90"] = (players["xassists"] * _per90).round(2)
players["GA_p90"] = (players["goals_plus_primary_assists"] * _per90).round(2)
players["xGA_p90"] = (players["xgoals_plus_xassists"] * _per90).round(2)
players["shots_p90"] = (players["shots"] * _per90).round(2)
players["sot_p90"] = (players["shots_on_target"] * _per90).round(2)
players["key_passes_p90"] = (players["key_passes"] * _per90).round(2)
players["ga_raw_p90"] = (players["ga_raw_total"] * _per90).round(3)

In [9]:
# Player attributes: age at season, multi-team flag, position group.
_birth = pd.to_datetime(players["birth_date"], errors="coerce")
players["age"] = (
    players["season_name"].astype("Int64") - _birth.dt.year.astype("Int64")
).astype("Int64")

players["is_multi_team"] = players["team_name"].str.contains("/", na=False)

_position_group_map: dict[str, str] = {
    "GK": "Goalkeeper",
    "CB": "Defender", "FB": "Defender",
    "DM": "Midfielder", "CM": "Midfielder", "AM": "Midfielder",
    "W": "Attacker", "ST": "Attacker",
}
players["position_group"] = players["general_position"].map(_position_group_map)

In [10]:
# Within-position-season percentile ranks for key per-90 + efficiency stats.
# Higher = better; computed within (general_position, season_name) cohort.
_rank_cols = [
    "G_p90", "xG_p90", "A_p90", "xA_p90", "GA_p90", "xGA_p90",
    "shots_p90", "key_passes_p90", "ga_raw_p90",
    "share_team_touches", "pass_completion_percentage",
    "passes_completed_over_expected_p100",
]
for _c in _rank_cols:
    players[f"{_c}_pct"] = (
        players.groupby(["general_position", "season_name"])[_c]
        .rank(pct=True)
        .round(3)
    )

# Final inf cleanup after all derived columns.
players = players.replace([float("inf"), float("-inf")], pd.NA)

In [11]:
_col_order = [
    # Identity / bio
    "player_id", "player_name", "birth_date", "age", "nationality", "HEIGHT",
    # Context
    "team_name", "is_multi_team", "season_name",
    "general_position", "position_group",
    "count_games", "minutes_played",
    # Shooting — volume
    "shots", "shots_on_target", "goals", "xgoals", "goals_minus_xgoals", "xplace",
    # Shooting — rates
    "GPerShot", "xGPerShot",
    "MinPerShot", "MinPerSOT", "MinPerG", "MinPerXG",
    "ShtPerSOT", "ShtPerG", "ShtPerXG",
    # Assists
    "key_passes", "primary_assists", "xassists", "primary_assists_minus_xassists",
    "MinPerAssist", "MinPerXA",
    # Combined
    "goals_plus_primary_assists", "xgoals_plus_xassists",
    "points_added", "xpoints_added",
    # Passing
    "attempted_passes", "PassAttemptsPerGame",
    "pass_completion_percentage", "xpass_completion_percentage",
    "passes_completed_over_expected", "passes_completed_over_expected_p100",
    "avg_distance_yds", "avg_vertical_distance_yds", "share_team_touches",
    # Goals Added — totals
    "ga_raw_total", "ga_aboveavg_total",
    # Goals Added — per action (raw)
    "ga_raw_dribbling", "ga_raw_fouling", "ga_raw_interrupting",
    "ga_raw_passing", "ga_raw_receiving", "ga_raw_shooting",
    # Goals Added — per action (above average)
    "ga_aboveavg_dribbling", "ga_aboveavg_fouling", "ga_aboveavg_interrupting",
    "ga_aboveavg_passing", "ga_aboveavg_receiving", "ga_aboveavg_shooting",
    # Goals Added — action counts
    "ga_actions_dribbling", "ga_actions_fouling", "ga_actions_interrupting",
    "ga_actions_passing", "ga_actions_receiving", "ga_actions_shooting",
    # Per-90 rates
    "G_p90", "xG_p90", "A_p90", "xA_p90", "GA_p90", "xGA_p90",
    "shots_p90", "sot_p90", "key_passes_p90", "ga_raw_p90",
    # Within-position-season percentile ranks
    "G_p90_pct", "xG_p90_pct", "A_p90_pct", "xA_p90_pct",
    "GA_p90_pct", "xGA_p90_pct",
    "shots_p90_pct", "key_passes_p90_pct", "ga_raw_p90_pct",
    "share_team_touches_pct", "pass_completion_percentage_pct",
    "passes_completed_over_expected_p100_pct",
]
players = players[_col_order]
players = players.sort_values(["player_name", "season_name"]).reset_index(drop=True)
display(players.head())

,player_id,player_name,birth_date,age,nationality,HEIGHT,team_name,is_multi_team,season_name,general_position,position_group,count_games,minutes_played,shots,shots_on_target,goals,xgoals,goals_minus_xgoals,xplace,GPerShot,xGPerShot,MinPerShot,MinPerSOT,MinPerG,MinPerXG,ShtPerSOT,ShtPerG,ShtPerXG,key_passes,primary_assists,xassists,primary_assists_minus_xassists,MinPerAssist,MinPerXA,goals_plus_primary_assists,xgoals_plus_xassists,points_added,xpoints_added,attempted_passes,PassAttemptsPerGame,pass_completion_percentage,xpass_completion_percentage,passes_completed_over_expected,passes_completed_over_expected_p100,avg_distance_yds,avg_vertical_distance_yds,share_team_touches,ga_raw_total,ga_aboveavg_total,ga_raw_dribbling,ga_raw_fouling,ga_raw_interrupting,ga_raw_passing,ga_raw_receiving,ga_raw_shooting,ga_aboveavg_dribbling,ga_aboveavg_fouling,ga_aboveavg_interrupting,ga_aboveavg_passing,ga_aboveavg_receiving,ga_aboveavg_shooting,ga_actions_dribbling,ga_actions_fouling,ga_actions_interrupting,ga_actions_passing,ga_actions_receiving,ga_actions_shooting,G_p90,xG_p90,A_p90,xA_p90,GA_p90,xGA_p90,shots_p90,sot_p90,key_passes_p90,ga_raw_p90,G_p90_pct,xG_p90_pct,A_p90_pct,xA_p90_pct,GA_p90_pct,xGA_p90_pct,shots_p90_pct,key_passes_p90_pct,ga_raw_p90_pct,share_team_touches_pct,pass_completion_percentage_pct,passes_completed_over_expected_p100_pct
0,p6qbOXVXQ0,A.J. Cochran,1993-02-09,24,USA,"6' 3""",STL,False,2017,CB,Defender,19,1810,7.0,2.0,1.0,0.59,0.4094,0.2253,0.14,0.084,258.6,905.0,1810.0,3064.68,3.5,7.0,11.85,2.0,0.0,0.0612,-0.0612,<NA>,29575.16,1.0,0.6519,0.4889,0.4078,910,47.89,0.7780,0.7741,3.6095,0.3966,25.3316,12.6953,0.0967,3.088,0.262,-0.209,-0.032,1.516,1.317,0.304,0.193,-0.256,0.178,0.256,0.182,-0.098,-0.002,187.0,19.0,291.0,910.0,643.0,7.0,0.05,0.03,0.0,0.0,0.05,0.03,0.35,0.1,0.1,0.154,0.733,0.459,0.409,0.179,0.635,0.321,0.5,0.361,0.74,0.740,0.291,0.338
1,p6qbOXVXQ0,A.J. Cochran,1993-02-09,25,USA,"6' 3""",ATL,False,2018,CB,Defender,30,2928,22.0,2.0,0.0,0.99,-0.9884,-0.6800,0.00,0.045,133.1,1464.0,<NA>,2962.36,11.0,<NA>,22.26,15.0,2.0,2.0524,-0.0524,1464.0,1426.62,2.0,3.0408,0.0000,0.8462,1926,64.20,0.8141,0.8040,19.5469,1.0149,24.6637,10.1808,0.1160,6.507,1.935,0.394,-0.386,2.965,2.636,0.395,0.501,0.320,-0.044,0.927,0.801,-0.254,0.186,416.0,52.0,523.0,1926.0,1450.0,22.0,0.0,0.03,0.06,0.06,0.06,0.09,0.68,0.06,0.46,0.2,0.336,0.528,0.906,0.936,0.692,0.817,0.825,0.928,0.875,0.956,0.628,0.467
2,p6qbOXVXQ0,A.J. Cochran,1993-02-09,26,USA,"6' 3""",PHX,False,2019,CB,Defender,24,2304,13.0,4.0,0.0,1.14,-1.1389,-0.8348,0.00,0.088,177.2,576.0,<NA>,2023.0,3.25,<NA>,11.41,6.0,4.0,1.2511,2.7489,576.0,1841.58,4.0,2.3900,0.0000,1.1139,1354,56.42,0.8242,0.8025,29.4010,2.1714,24.8520,11.6658,0.1061,3.026,-0.571,0.089,-0.346,0.105,2.289,0.711,0.177,0.030,-0.077,-1.498,0.845,0.200,-0.071,332.0,41.0,350.0,1354.0,985.0,13.0,0.0,0.04,0.16,0.05,0.16,0.09,0.51,0.16,0.23,0.118,0.293,0.601,0.991,0.891,0.92,0.764,0.618,0.647,0.345,0.894,0.586,0.759
3,p6qbOXVXQ0,A.J. Cochran,1993-02-09,27,USA,"6' 3""",PHX,False,2020,CB,Defender,13,1152,7.0,2.0,2.0,0.56,1.4422,0.5648,0.29,0.080,164.6,576.0,576.0,2065.26,3.5,3.5,12.55,2.0,0.0,0.1065,-0.1065,<NA>,10816.9,2.0,0.6643,0.2323,0.2944,650,50.00,0.8338,0.8253,5.5866,0.8595,26.0826,10.0659,0.1016,1.622,-0.177,0.027,-0.133,0.381,0.706,0.432,0.209,-0.002,0.001,-0.421,-0.016,0.177,0.085,109.0,12.0,150.0,650.0,501.0,7.0,0.16,0.04,0.0,0.01,0.16,0.05,0.55,0.16,0.16,0.127,0.933,0.613,0.412,0.567,0.891,0.496,0.697,0.577,0.479,0.775,0.620,0.500
4,p6qbOXVXQ0,A.J. Cochran,1993-02-09,28,USA,"6' 3""",IND,False,2021,CB,Defender,21,1834,8.0,2.0,0.0,0.36,-0.3626,-0.0809,0.00,0.045,229.2,917.0,<NA>,5057.92,4.0,<NA>,22.06,4.0,0.0,0.2492,-0.2492,<NA>,7359.55,0.0,0.6118,0.0000,0.3103,917,43.67,0.8321,0.8143,16.3027,1.7778,24.5193,10.5190,0.0841,1.563,-1.301,0.081,-0.348,0.249,1.177,0.228,0.176,0.034,-0.134,-1.028,0.027,-0.179,-0.022,152.0,17.0,273.0,917.0,642.0,8.0,0.0,0.02,0.0,0.01,0.0,0.03,0.39,0.1,0.2,0.077,0.272,0.279,0.4,0.455,

## Goalkeepers

In [12]:
gk_players = players_merge.merge(gk_xg, on="player_id", how="left")
gk_players = gk_players[gk_players["season_name"].notna()]
gk_players = gk_players[gk_players["shots_faced"] > 0]
gk_players = gk_players[gk_players["minutes_played"] >= GK_MIN_MINUTES]  # excludes backup/emergency appearances

gk_players["season_name"] = gk_players["season_name"].astype("Int64")
gk_players["minutes_played"] = gk_players["minutes_played"].astype("Int64")

gk_players["MINUTES PER SHOT"] = (
    gk_players["minutes_played"] / gk_players["shots_faced"]
).round(1)
gk_players["MINUTES PER GOAL"] = (
    gk_players["minutes_played"] / gk_players["goals_conceded"]
).round(1)
gk_players["MINUTES PER SAVE"] = (
    gk_players["minutes_played"] / gk_players["saves"]
).round(1)
gk_players["MINUTES PER XG"] = (
    gk_players["minutes_played"] / gk_players["xgoals_gk_faced"]
).round(1)
gk_players["SHOTS PER GOAL"] = (
    gk_players["shots_faced"] / gk_players["goals_conceded"]
).round(2)
gk_players["SAVES PER GOAL"] = (
    gk_players["saves"] / gk_players["goals_conceded"]
).round(1)
gk_players["XG PER GOAL"] = (
    gk_players["xgoals_gk_faced"] / gk_players["goals_conceded"]
).round(2)
gk_players["XG PER SHOT"] = (
    gk_players["xgoals_gk_faced"] / gk_players["shots_faced"]
).round(2)

# GKs with 0 goals_conceded or 0 saves produce inf in several rate columns.
gk_players = gk_players.replace([float("inf"), float("-inf")], pd.NA)

In [13]:
# Merge in flattened GK Goals Added (per-action + totals).
_gk_ga_join = ["player_id", "season_name"]
_gk_ga_extra = [c for c in gk_ga.columns if c not in gk_players.columns]
gk_players = gk_players.merge(
    gk_ga[_gk_ga_join + _gk_ga_extra], on=_gk_ga_join, how="left"
)

In [14]:
# GK derived metrics: save %, count_games estimate, per-90 rates, attributes, and percentile ranks.
gk_players["save_pct"] = (
    gk_players["saves"] / gk_players["shots_faced"]
).round(3)

gk_players["count_games_est"] = (gk_players["minutes_played"] / 90).round(1)

_per90 = 90 / gk_players["minutes_played"]
gk_players["goals_conceded_p90"] = (gk_players["goals_conceded"] * _per90).round(2)
gk_players["saves_p90"] = (gk_players["saves"] * _per90).round(2)
gk_players["shots_faced_p90"] = (gk_players["shots_faced"] * _per90).round(2)
gk_players["xgoals_faced_p90"] = (gk_players["xgoals_gk_faced"] * _per90).round(2)
gk_players["ga_raw_p90"] = (gk_players["ga_raw_total"] * _per90).round(3)

# Attributes mirroring outfield frame.
_gk_birth = pd.to_datetime(gk_players["birth_date"], errors="coerce")
gk_players["age"] = (
    gk_players["season_name"].astype("Int64") - _gk_birth.dt.year.astype("Int64")
).astype("Int64")

gk_players["is_multi_team"] = gk_players["team_name"].str.contains("/", na=False)

# Within-season percentile ranks. All GKs share one position so no position grouping needed.
# Higher is better for save metrics; lower is better for conceded/xG metrics.
_gk_rank_high = ["save_pct", "saves_p90", "ga_raw_p90"]
_gk_rank_low = ["goals_conceded_p90", "goals_minus_xgoals_gk", "goals_divided_by_xgoals_gk"]
for _c in _gk_rank_high:
    gk_players[f"{_c}_pct"] = (
        gk_players.groupby("season_name")[_c].rank(pct=True).round(3)
    )
for _c in _gk_rank_low:
    gk_players[f"{_c}_pct"] = (
        gk_players.groupby("season_name")[_c].rank(pct=True, ascending=False).round(3)
    )

gk_players = gk_players.replace([float("inf"), float("-inf")], pd.NA)

In [15]:
_col_order = [
    # Identity / bio
    "player_id", "player_name", "birth_date", "age", "nationality", "HEIGHT",
    # Context
    "team_name", "is_multi_team", "season_name", "minutes_played", "count_games_est",
    # Volume
    "shots_faced", "goals_conceded", "saves", "share_headed_shots",
    # xG
    "xgoals_gk_faced", "goals_minus_xgoals_gk", "goals_divided_by_xgoals_gk",
    # Rate stats
    "save_pct",
    "MINUTES PER SHOT", "MINUTES PER GOAL", "MINUTES PER SAVE", "MINUTES PER XG",
    "SHOTS PER GOAL", "SAVES PER GOAL", "XG PER GOAL", "XG PER SHOT",
    # Per-90
    "goals_conceded_p90", "saves_p90", "shots_faced_p90",
    "xgoals_faced_p90", "ga_raw_p90",
    # Within-season percentile ranks
    "save_pct_pct", "saves_p90_pct", "ga_raw_p90_pct",
    "goals_conceded_p90_pct", "goals_minus_xgoals_gk_pct", "goals_divided_by_xgoals_gk_pct",
    # Goals Added — totals
    "ga_raw_total", "ga_aboveavg_total",
    # Goals Added — per action (raw)
    "ga_raw_claiming", "ga_raw_fielding", "ga_raw_handling",
    "ga_raw_passing", "ga_raw_shotstopping", "ga_raw_sweeping",
    # Goals Added — per action (above average)
    "ga_aboveavg_claiming", "ga_aboveavg_fielding", "ga_aboveavg_handling",
    "ga_aboveavg_passing", "ga_aboveavg_shotstopping", "ga_aboveavg_sweeping",
    # Goals Added — action counts
    "ga_actions_claiming", "ga_actions_fielding", "ga_actions_handling",
    "ga_actions_passing", "ga_actions_shotstopping", "ga_actions_sweeping",
]
gk_players = gk_players[_col_order]
gk_players = gk_players.sort_values(["player_name", "season_name"]).reset_index(drop=True)
display(gk_players.head())

,player_id,player_name,birth_date,age,nationality,HEIGHT,team_name,is_multi_team,season_name,minutes_played,count_games_est,shots_faced,goals_conceded,saves,share_headed_shots,xgoals_gk_faced,goals_minus_xgoals_gk,goals_divided_by_xgoals_gk,save_pct,MINUTES PER SHOT,MINUTES PER GOAL,MINUTES PER SAVE,MINUTES PER XG,SHOTS PER GOAL,SAVES PER GOAL,XG PER GOAL,XG PER SHOT,goals_conceded_p90,saves_p90,shots_faced_p90,xgoals_faced_p90,ga_raw_p90,save_pct_pct,saves_p90_pct,ga_raw_p90_pct,goals_conceded_p90_pct,goals_minus_xgoals_gk_pct,goals_divided_by_xgoals_gk_pct,ga_raw_total,ga_aboveavg_total,ga_raw_claiming,ga_raw_fielding,ga_raw_handling,ga_raw_passing,ga_raw_shotstopping,ga_raw_sweeping,ga_aboveavg_claiming,ga_aboveavg_fielding,ga_aboveavg_handling,ga_aboveavg_passing,ga_aboveavg_shotstopping,ga_aboveavg_sweeping,ga_actions_claiming,ga_actions_fielding,ga_actions_handling,ga_actions_passing,ga_actions_shotstopping,ga_actions_sweeping
0,7VqG3Onxqv,AJ Marcucci,1999-07-31,22,USA,"6' 4""",NYRB,False,2021,1887,21.0,104.0,37.0,67.0,0.1058,41.3874,-4.3874,0.8940,0.644,18.1,51.0,28.2,45.6,2.81,1.8,1.12,0.40,1.76,3.2,4.96,1.97,0.27,0.457,0.79,0.84,0.296,0.901,0.753,5.652,5.162,0.183,-0.059,0.062,1.361,4.387,-0.282,0.173,0.057,0.097,0.109,4.949,-0.224,29.0,364.0,67.0,683.0,104.0,56.0
1,7VqG3Onxqv,AJ Marcucci,1999-07-31,23,USA,"6' 4""",NYRB,False,2022,1337,14.9,73.0,25.0,47.0,0.0685,25.6958,-0.6958,0.9729,0.644,18.3,53.5,28.4,52.0,2.92,1.9,1.03,0.35,1.68,3.16,4.91,1.73,0.096,0.449,0.768,0.638,0.348,0.667,0.594,1.425,1.077,0.070,-0.544,0.470,1.169,0.696,-0.437,0.063,-0.461,0.495,0.282,1.094,-0.396,21.0,279.0,47.0,525.0,72.0,57.0
2,4wM4J2pBqj,Aaron Cervantes,2002-03-20,17,USA,"5' 11""",OC,False,2019,1080,12.0,36.0,14.0,21.0,0.0833,13.0072,0.9928,1.0763,0.583,30.0,77.1,51.4,83.0,2.57,1.5,0.93,0.36,1.17,1.75,3.0,1.08,-0.076,0.215,0.086,0.403,0.715,0.473,0.430,-0.914,-1.195,0.032,0.008,0.054,0.477,-0.993,-0.492,0.026,0.074,0.074,-0.240,-0.671,-0.458,8.0,132.0,21.0,291.0,35.0,18.0
3,4wM4J2pBqj,Aaron Cervantes,2002-03-20,18,USA,"5' 11""",OC,False,2020,545,6.1,21.0,6.0,15.0,0.0952,6.0491,-0.0491,0.9919,0.714,26.0,90.8,36.3,90.1,3.5,2.5,1.01,0.29,0.99,2.48,3.47,1.0,0.081,0.729,0.507,0.667,0.75,0.611,0.639,0.491,0.349,0.041,-0.059,0.182,0.166,0.049,0.112,0.038,-0.026,0.192,-0.195,0.211,0.128,13.0,64.0,15.0,166.0,21.0,9.0
4,gpMOo141Mz,Abraham Rodriguez,2002-07-15,17,USA,"5' 9""",COS,False,2019,1557,17.3,94.0,37.0,55.0,0.1915,32.3515,4.6485,1.1437,0.585,16.6,42.1,28.3,48.1,2.54,1.5,0.87,0.34,2.14,3.18,5.43,1.87,-0.253,0.226,0.71,0.215,0.161,0.086,0.280,-4.385,-4.790,0.055,-0.005,-0.193,1.027,-4.648,-0.621,0.047,0.091,-0.164,-0.006,-4.185,-0.573,16.0,235.0,55.0,530.0,92.0,53.0


In [16]:
players.to_parquet("data/players.parquet", index=False)
gk_players.to_parquet("data/gk_players.parquet", index=False)